In [1]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

In [2]:
mat = material("body")
prob = Problem([mat])

Problem("I-szelveny-2", :Solid, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 112833, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [3]:
n1x(x, y, z) = x
n1y(x, y, z) = y
n2x(x, y, z) = -y
n2y(x, y, z) = x
cs = CoordinateSystem([n1x, n1y, 0], [n2x, n2y, 0])
Q = rotateNodes(prob, "body", cs)

LowLevelFEM.Transformation(sparse([1, 2, 1, 2, 3, 4, 5, 4, 5, 6  …  338494, 338495, 338494, 338495, 338496, 338497, 338498, 338497, 338498, 338499], [1, 1, 2, 2, 3, 4, 4, 5, 5, 6  …  338494, 338494, 338495, 338495, 338496, 338497, 338497, 338498, 338498, 338499], [-0.4472135954999579, -0.8944271909999159, 0.8944271909999159, -0.4472135954999579, 0.9999999999999999, 0.4472135954999579, -0.8944271909999159, 0.8944271909999159, 0.4472135954999579, 0.9999999999999999  …  0.3804360740512166, -0.924807219673645, 0.924807219673645, 0.3804360740512166, 1.0000000000000002, -0.021092324015402158, 0.999777532187851, -0.999777532187851, -0.021092324015402158, 1.0000000000000002], 338499, 338499), 112833, 3)

In [4]:
M0 = integrate(prob, "right", (x, y, z) -> (x^2 + y^2))

2.460497060569796e7

In [5]:
supp1 = displacementConstraint("left", uy=0)

fy(x, y, z) = √(x^2 + y^2) / M0 * 10000
load1 = load("right", fy=fy)

elsupp1 = elasticSupport("left", kz=1)
elsupp2 = elasticSupport("right", kx=1)

BoundaryCondition("right", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:kx => 1))

In [6]:
f = loadVector(prob, [load1]);

In [7]:
showDoFResults(Q * f, name="load")

0

In [8]:
K = stiffnessMatrix(prob)

sparse([1, 2, 3, 115, 116, 117, 754, 755, 756, 757  …  328512, 331114, 331115, 331116, 333739, 333740, 333741, 338497, 338498, 338499], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499], [247285.12538616545, 95148.24721188965, 85813.15567468441, -63211.41046427257, 25601.128729018656, 26441.31459153176, 15188.0584981424, 9915.464321528003, 21294.786939598212, 16218.701775497684  …  -65887.44308570187, 37774.79330996382, -176999.1084781225, 114329.64593834887, 50699.52318257713, 46333.919351545774, 104317.77080235285, -135342.4472554631, 278146.97906030354, 3.3278486162643894e6], 338499, 338499)

In [9]:
C1 = elasticSupportMatrix(prob, [elsupp1])

sparse([3, 117, 756, 12942, 6, 198, 201, 12948, 9, 204  …  12549, 12555, 12915, 13242, 15, 396, 12552, 12558, 12912, 13245], [3, 3, 3, 3, 6, 6, 6, 6, 9, 9  …  13242, 13242, 13242, 13242, 13245, 13245, 13245, 13245, 13245, 13245], [1.3764168787330888, 0.3476503405740741, 0.34055809879247034, 0.6882084393665453, 1.3764168787331617, 0.34765034057409866, 0.34055809879248217, 0.6882084393665817, 1.3787715859145613, 0.3407581356971643  …  0.5714160071252107, 0.5925075891011984, 0.5019634998170357, 2.9458000071129256, 0.653960774941222, 0.6259521361282291, 0.5714160071252159, 0.5925075891011765, 0.5019634998170325, 2.945800007112875], 338499, 338499)

In [10]:
C2 = elasticSupportMatrix(prob, [elsupp2])
C2 = Q * C2 * Q'

sparse([73, 74, 7363, 7364, 7876, 7877, 229582, 229583, 73, 74  …  11182, 11183, 229192, 229193, 229198, 229199, 229552, 229553, 229885, 229886], [73, 73, 73, 73, 73, 73, 73, 73, 74, 74  …  229886, 229886, 229886, 229886, 229886, 229886, 229886, 229886, 229886, 229886], [0.27528337574661776, 0.5505667514932355, 0.06561460955147866, 0.14094990199947266, 0.06997112671336576, 0.13527751164584048, 0.13481679717687947, 0.2766777952818681, 0.5505667514932355, 1.101133502986471  …  0.058415886954107654, 0.6213326157845992, 0.06181708892316458, 0.5663364863106989, 0.03058237046773353, 0.589936330457554, 0.04079695367447973, 0.4987906327092252, 0.2274251283887721, 2.928136146405392], 338499, 338499)

In [11]:
q = solveDisplacement(Q' * (K + C1 + C2) * Q, f, support=[supp1])

nodal VectorField
[0.00013783407476454442; 0.0; … ; 0.03744206079114293; -0.00011281877744301698;;]

In [12]:
#K1, f1 = applyBoundaryConditions(Q' * (K + C1 + C2) * Q, f, [supp1])

In [13]:
#q = (Q * K1 * Q') \ (Q * f1);

In [14]:
showDoFResults(q, :uvec, name="displacement");

In [15]:
S = solveStress(q, DoFResults=true);

In [16]:
showDoFResults(S, :syz)
showDoFResults(S, :sx)
showDoFResults(S, :sy)
showDoFResults(S, :sz)
showDoFResults(S, :sxy)
showDoFResults(S, :szx)

#plotOnPath("path", syz)

7

In [17]:
R = nodePositionVector(prob)
R = nodesToElements(R, onPhysicalGroup="left")

n2 = normalVector(prob, "left")

S2 = nodesToElements(S, onPhysicalGroup="left")

τ2 = S2 * n2;

In [18]:
showElementResults(τ2, name="shear stress")

8

In [19]:
M2 = R × (S2 * n2);

In [20]:
ez = VectorField(prob, [field("left", fx=0, fy=0, fz=1)]);

In [21]:
M2z = M2 * ez;

In [22]:
integrate(prob, "left", M2z)

-1.2872720189322592e6

In [23]:
umax = probe(q, 0, 100, 1000)[1]

5.74e-8

In [24]:
umax / 100 / 1000 * 2e5 / 2 / (1 + 0.3) * 10^3 * (180 + 100 + 100) / 3

0.005592820512820513

In [25]:
n1 = normalVector(prob, "mid");

In [26]:
showDoFResults(n1, name="normal vector")

9

In [27]:
S2 = nodesToElements(S, onPhysicalGroup="mid")
showElementResults(S2 * n1, name="shear stress mid")

10

In [28]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
